# Galaxy Bias from Threshold Crossing

A 2D Gaussian overdensity field $\delta$ with a threshold sweeping up and down through it.
Wherever the field exceeds the threshold we call the region a *tracer*, and we measure the
correlation function of those tracers against the correlation function of the field itself.

The point of the visualization: **the tracer correlation function is a simple multiple of the
field's**, and the multiple is set entirely by how rare the tracers are.

## The physics

For a Gaussian field with rms $\sigma$, select the *excursion set* $\delta > \nu\sigma$.
Kaiser (1984) showed the correlation function of that set is, exactly,

$$\xi_\nu(r) = \sum_{j=1}^{\infty}
  \frac{\left[\varphi(\nu)\,He_{j-1}(\nu)\right]^2}{j!\,\left[1-\Phi(\nu)\right]^2}\,w(r)^j,
  \qquad w \equiv \xi_\delta(r)/\sigma^2$$

with $\varphi$, $\Phi$ the standard normal pdf and CDF and $He_j$ the probabilists' Hermite
polynomials. Keeping only $j=1$ gives linear bias:

$$\boxed{\;\xi_\nu(r) \;\to\; b^2(\nu)\,\xi_\delta(r),
  \qquad b(\nu) = \frac{1}{\sigma}\frac{\varphi(\nu)}{1-\Phi(\nu)}\;}$$

i.e. $b$ is the inverse Mills ratio over $\sigma$, and $b \to \nu/\sigma$ for rare tracers.
Linear bias is an **asymptotic** statement: it holds where $\xi_\delta \ll \sigma^2$. At small
separations the higher-$j$ terms take over and the ratio $\xi_\nu/\xi_\delta$ climbs above $b^2$.
The animation shows both regimes at once.

**On the smoothing scale.** Bias depends on the smoothing scale $R$ only through
$\nu = \delta_c/\sigma(R)$, so sweeping the threshold at fixed $R$ is the same operation as
moving to rarer, more massive halos. What is dropped is the $\nu \leftrightarrow M$ mapping and
the scale-dependent (peak/$k^2$) corrections, not the core of the effect.

In [1]:
import numpy as np
import matplotlib
import matplotlib.pyplot as pl
import matplotlib.animation as animation
from matplotlib.gridspec import GridSpec
from matplotlib.colors import ListedColormap
from IPython.display import HTML

from scipy.stats import norm
from scipy.special import factorial
from scipy.ndimage import label, center_of_mass
from numpy.polynomial.hermite_e import hermeval

In [ ]:
def use_style(dark=True):
    '''dark_theme sets keys custom.mplstyle never overrides (text.color,
    savefig.facecolor, ...), so reset to defaults before re-applying.'''
    pl.rcdefaults()
    pl.style.use(['custom.mplstyle', 'dark_theme.mplstyle'] if dark else ['custom.mplstyle'])
    pl.rcParams['animation.embed_limit'] = 80.0   # MB, for to_jshtml()


use_style(dark=True)

## Field setup

A periodic 2D Gaussian random field, $P(k) \propto k^{n}$ with Gaussian smoothing on scale
$R_{\rm smooth}$, normalized to $\sigma = 1$ so that $\nu$ is directly the threshold value.
The box must be large compared to $R_{\rm smooth}$ for a clean linear-bias plateau to exist.

In [3]:
N        = 2048     # grid size [pixels]
NREAL    = 12       # realizations averaged into the measured xi
NSPEC    = -1.0     # P(k) ~ k^NSPEC
RSMOOTH  = 4.0      # Gaussian smoothing scale [pixels]
SEED     = 20260803
NBIN     = 16       # radial bins for xi(r)
JMAX     = 60       # terms kept in the Kaiser series
CUT      = 384      # size of the displayed cutout [pixels]

nus = np.linspace(-0.5, 2.5, 61)   # threshold values swept by the animation

In [4]:
def bias(nu):
    '''Kaiser threshold bias b(nu) for a Gaussian field with sigma = 1.'''
    return norm.pdf(nu) / norm.sf(nu)


def kaiser_series(nu, w, jmax=JMAX):
    '''Exact xi of the excursion set delta > nu, for a Gaussian field with sigma = 1.'''
    j = np.arange(1, jmax + 1)
    he = np.array([hermeval(nu, [0] * (n - 1) + [1]) for n in j])   # He_{j-1}(nu)
    coef = (norm.pdf(nu) * he) ** 2 / (factorial(j) * norm.sf(nu) ** 2)
    return np.sum(coef[:, None] * w[None, :] ** j[:, None], axis=0)

In [5]:
# --- Fourier grid and the sqrt of the power spectrum used to colour white noise
kx = np.fft.fftfreq(N)[:, None]
ky = np.fft.rfftfreq(N)[None, :]
kk = np.sqrt(kx**2 + ky**2)
kk[0, 0] = 1.0                                    # avoid 0**negative

pk = kk**NSPEC * np.exp(-(kk * RSMOOTH * 2 * np.pi)**2)
pk[0, 0] = 0.0                                    # no DC mode
sqrt_pk = np.sqrt(pk)

# --- radial binning of the real-space separation grid
ix   = np.fft.fftfreq(N) * N
sep  = np.sqrt(ix[:, None]**2 + ix[None, :]**2)
rbin = np.geomspace(RSMOOTH, N / 8, NBIN + 1)

ibin  = np.digitize(sep, rbin) - 1
valid = (ibin >= 0) & (ibin < NBIN)
counts = np.bincount(ibin[valid], minlength=NBIN)
rmid   = np.bincount(ibin[valid], weights=sep[valid], minlength=NBIN) / counts


def xi_of(f):
    '''Correlation function of a zero-mean periodic field, radially binned via FFT.'''
    corr = np.fft.irfft2(np.abs(np.fft.rfft2(f))**2, s=(N, N)) / f.size
    return np.bincount(ibin[valid], weights=corr[valid], minlength=NBIN) / counts

In [6]:
rng = np.random.default_rng(SEED)

fields = []
xi_field = np.zeros(NBIN)

for i in range(NREAL):
    g = np.fft.irfft2(np.fft.rfft2(rng.standard_normal((N, N))) * sqrt_pk, s=(N, N))
    g /= g.std()                       # enforce sigma = 1 exactly
    fields.append(g.astype(np.float32))
    xi_field += xi_of(g)

xi_field /= NREAL

print(f"{NREAL} realizations of {N}^2, sigma = 1")
print(f"xi_field spans {xi_field.min():.4f} .. {xi_field.max():.4f}"
      f" over r = {rmid[0]:.1f} .. {rmid[-1]:.1f} px")

12 realizations of 2048^2, sigma = 1
xi_field spans 0.0123 .. 0.8515 over r = 4.6 .. 228.0 px


## Measure $\xi$ of the thresholded field

For each threshold we build the 0/1 excursion-set mask and take its correlation function.
Using the **mask** rather than a point catalogue is deliberate: it is exactly the quantity the
Kaiser formula predicts, with no shot-noise term to subtract and no exclusion effect from
finite tracer size. Markers are drawn at region centroids in the image panel purely for looks.

In [7]:
xi_tracer = np.zeros((len(nus), NBIN))

for i, nu in enumerate(nus):
    acc = np.zeros(NBIN)
    for g in fields:
        m = (g > nu).astype(np.float64)
        acc += xi_of(m / m.mean() - 1.0)
    xi_tracer[i] = acc / NREAL

print("done")

done


In [8]:
# sanity check: the large-r plateau should approach b^2(nu)
plateau = rmid > N / 20
print(" nu     b(nu)    b^2      measured plateau   ratio")
for i in range(0, len(nus), 10):
    meas = (xi_tracer[i] / xi_field)[plateau].mean()
    b2 = bias(nus[i])**2
    print(f"{nus[i]:5.2f}  {bias(nus[i]):7.3f}  {b2:7.3f}  {meas:16.3f}  {meas/b2:7.3f}")

 nu     b(nu)    b^2      measured plateau   ratio
-0.50    0.509    0.259             0.259    0.997
 0.00    0.798    0.637             0.639    1.003
 0.50    1.141    1.302             1.308    1.005
 1.00    1.525    2.326             2.411    1.036
 1.50    1.939    3.758             3.979    1.059
 2.00    2.373    5.632             5.899    1.047
 2.50    2.823    7.968             7.563    0.949


## The animation

In [9]:
DARK = dict(field='#4CC9FE', tracer='#FADA7A', lin='white', exact='#d25dd4',
            thresh='#fa8174', cmap='bone', dim=(0.02, 0.02, 0.06, 0.78), mec='0.15')

LIGHT = dict(field='#309898', tracer='#FF9F00', lin='black', exact='#A53860',
             thresh='#d62728', cmap='bone_r', dim=(1.0, 1.0, 1.0, 0.78), mec='0.25')

xgrid = np.linspace(-4, 4, 400)
frames = np.concatenate([np.arange(len(nus)), np.arange(len(nus) - 2, 0, -1)])

In [10]:
def build_animation(C, cut_field):
    '''Assemble the four-panel figure and its FuncAnimation for one colour scheme.'''

    fig = pl.figure(figsize=(14.2, 7.4))
    outer = GridSpec(1, 2, width_ratios=[0.76, 1.0], wspace=0.20,
                     left=0.035, right=0.975, top=0.905, bottom=0.085)
    gl = outer[0].subgridspec(2, 1, height_ratios=[1, 0.26], hspace=0.13)
    gr = outer[1].subgridspec(2, 1, hspace=0.34)

    axf = fig.add_subplot(gl[0])    # the field + tracers
    axp = fig.add_subplot(gl[1])    # pdf of delta with the sweeping threshold
    axx = fig.add_subplot(gr[0])    # xi(r)
    axr = fig.add_subplot(gr[1])    # xi_tr / (b^2 xi_field)

    dim_cmap = ListedColormap([C['dim'], (0, 0, 0, 0)])

    def draw(frame):
        i = frames[frame]
        nu, b2 = nus[i], bias(nus[i])**2
        ser = kaiser_series(nus[i], xi_field)

        for ax in (axf, axp, axx, axr):
            ax.clear()

        # --- field panel: everything below threshold is dimmed out
        mask = cut_field > nu
        axf.imshow(cut_field, cmap=C['cmap'], origin='lower', vmin=-3.2, vmax=3.2,
                   interpolation='bilinear')
        axf.imshow(mask.astype(float), cmap=dim_cmap, origin='lower', vmin=0, vmax=1)
        axf.contour(mask.astype(float), levels=[0.5], colors=[C['thresh']], linewidths=1.6)

        lab, n = label(mask)
        if 0 < n < 4000:
            cm = np.array(center_of_mass(mask, lab, np.arange(1, n + 1)))
            axf.plot(cm[:, 1], cm[:, 0], 'o', color=C['lin'], ms=6.0, mew=1.2,
                     mec=C['thresh'], ls='none')
        axf.set_xticks([]); axf.set_yticks([])
        axf.set_title(rf"$\nu = {nu:+.2f}$   |   $b(\nu) = {bias(nu):.2f}$   |   "
                      rf"$f_{{>\nu}} = {norm.sf(nu) * 100:.1f}\%$   |   "
                      rf"$N_{{\rm regions}} = {n}$", fontsize=15, fontweight='normal')

        # --- pdf strip
        axp.plot(xgrid, norm.pdf(xgrid), color=C['field'], lw=2.5)
        axp.fill_between(xgrid, 0, norm.pdf(xgrid), where=xgrid > nu,
                         color=C['tracer'], alpha=0.6, lw=0)
        axp.axvline(nu, color=C['thresh'], lw=2.5)
        axp.set_yticks([]); axp.set_xlim(-4, 4); axp.set_ylim(0, 0.44)
        axp.set_xlabel(r"$\delta / \sigma$", fontsize=13, labelpad=2)

        # --- xi(r)
        axx.loglog(rmid, xi_field, color=C['field'], lw=2.5, label=r"$\xi_\delta$   field")
        axx.loglog(rmid, ser, color=C['exact'], lw=2.2, label="Kaiser series (exact)")
        axx.loglog(rmid, b2 * xi_field, color=C['lin'], lw=2, ls='--',
                   label=r"$b^2 \xi_\delta$   linear bias")
        axx.loglog(rmid, xi_tracer[i], 'o', color=C['tracer'], ms=7.5, mew=0.8,
                   mec=C['mec'], label=r"$\xi_{\rm tr}$   measured")
        axx.set_ylim(2e-3, 5e2)
        axx.set_xlabel("r  [pixels]"); axx.set_ylabel(r"$\xi(r)$")
        axx.legend(frameon=False, fontsize=11.5, loc='upper right')

        # --- ratio, normalized so linear bias sits at unity
        axr.plot(rmid, xi_tracer[i] / (b2 * xi_field), 'o', color=C['tracer'],
                 ms=7.5, mew=0.8, mec=C['mec'], label='measured')
        axr.plot(rmid, ser / (b2 * xi_field), color=C['exact'], lw=2.2)
        axr.axhline(1.0, color=C['lin'], ls='--', lw=2)
        axr.set_xscale('log'); axr.set_yscale('log'); axr.set_ylim(0.55, 40)
        axr.set_xlabel("r  [pixels]")
        axr.set_ylabel(r"$\xi_{\rm tr} \,/\, b^2 \xi_\delta$")
        axr.text(rmid[0], 0.75, "linear bias exact", color=C['lin'],
                 fontsize=12, ha='left', va='center')

        return ()

    anim = animation.FuncAnimation(fig=fig, func=draw, frames=len(frames), interval=90)
    return fig, anim

In [ ]:
cut_field = fields[0][:CUT, :CUT].astype(np.float64)

fig, anim = build_animation(DARK, cut_field)
HTML(anim.to_jshtml())

In [12]:
anim.save("assets/galaxy_bias_dark.gif", writer='pillow', fps=12, dpi=80)
pl.close(fig)

In [13]:
use_style(dark=False)
fig_l, anim_l = build_animation(LIGHT, cut_field)
anim_l.save("assets/galaxy_bias.gif", writer='pillow', fps=12, dpi=80)
pl.close(fig_l)

use_style(dark=True)

In [14]:
# MP4 versions -- requires ffmpeg on PATH (conda install ffmpeg)
if 'ffmpeg' in animation.writers.list():
    fig, anim = build_animation(DARK, cut_field)
    anim.save("assets/galaxy_bias_dark.mp4", fps=12, dpi=100)
    pl.close(fig)

    use_style(dark=False)
    fig_l, anim_l = build_animation(LIGHT, cut_field)
    anim_l.save("assets/galaxy_bias.mp4", fps=12, dpi=100)
    pl.close(fig_l)
    use_style(dark=True)
else:
    print("ffmpeg not available -- GIF only")

ffmpeg not available -- GIF only


## What the animation shows, and what it glosses over

**Shows.** As the threshold rises the tracers become rarer and more strongly clustered. On large
scales $\xi_{\rm tr}$ is a rigid vertical shift of $\xi_\delta$ on the log-log plot, by exactly
$b^2(\nu)$ — the bottom panel sits at unity there. On small scales it lifts off, and the full
Kaiser series tracks that lift-off to within the measurement noise.

**Glosses over.**

- *Percolation at low $\nu$.* Below $\nu \approx 0.5$ the excursion set connects into a few
  spanning regions, so the centroid markers stop meaning anything. The mask-based $\xi$ is still
  exact; only the "discrete tracer" reading of the picture breaks.
- *Peaks vs. excursion sets.* Real halos sit at peaks, whose bias carries extra
  $k^2$-dependent terms (BBKS). These agree with $b(\nu)$ at large $r$ but not near $R_{\rm smooth}$.
- *Gaussianity.* The exact series needs a Gaussian field. A lognormal field is more realistic and,
  because the transform is monotonic, selects the *identical* excursion set — but then $\xi_\delta$
  becomes $e^{\xi_g} - 1$ and the clean series no longer applies.
- *Noise at high $\nu$.* At $\nu = 2.5$ only ~0.6% of the area is above threshold; the outermost
  bins are visibly noisy even averaged over 12 realizations.